## TCP server class

In [22]:
import socket
import numpy as np
import os

class tcpsocket:
    def __init__(self, address):
        
        self.sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self.sock.bind(('localhost', address))
        self.sock.listen(1)
        print('waiting for unity connection...') 
        
        self.conn, self.addr = self.sock.accept()
        print('unity connected!')

    def getresponse(self):
        data = self.conn.recv(1024)
        if not data:
            raise Exception("Empty message from Unity, socket disconnected")
        out = np.frombuffer(data,dtype='<f')
        #print("Python received:", out)
        return out

    #placeholder message
    def send(self):
        self.conn.sendall(b'aaaaa')

    #data message
    def sendvector(self,data):
        message = data.tobytes() #this is byte[]
        self.conn.sendall(message)

    def close(self):
        self.conn.close()
        self.sock.close()


## Frame grabber class

In [23]:
import dxcam 
import mss 
import time
import cv2 #just for plotting images, not grabbing anything

#fastest, but windows only
class frame_dx:
    
    def __init__(self,region):
        self.region = region
        self.cam = dxcam.create(region=region,output_color="GRAY")

    def grab(self):
        
        img = self.cam.grab()
        
        #if windows hasn't flipped to a new frame yet, dxcam returns None. In that case, wait for a new one.
        while img is None:
            img = self.cam.grab()
            time.sleep(0.0005)
        
        frame = img[:,:,0].squeeze() #frames are grabbed with 3 grayscale channels, keep only one.  
        
        return frame

    def close(self):
        cv2.destroyAllWindows()
        self.cam.release()

#cross platform, but slow
class frame_mss:
    
    def __init__(self,region):
        self.region = region
        self.cam = mss.mss()

    def grab(self):
        img = np.asarray(self.cam.grab(self.region))
        frame = cv2.cvtColor(img, cv2.COLOR_RGBA2GRAY)
        return frame

    def close(self):
        cv2.destroyAllWindows()
        self.cam.close()

## Set up pytorch model

In [24]:
import torch
from torch import nn
import random

#seeding
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed = 25 #(the highest number)
set_seed(seed)

torch.backends.cudnn.benchmark = True

### Model class

In [25]:
#get device for training
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

class FullyConnected(nn.Module):
    def __init__(self,input_dims):
        
        hidden_size = 512
        input_size = input_dims[0] * input_dims[1]
        output_size = 3
        
        super().__init__()
        self.flatten = nn.Flatten()
        self.fully_connected = nn.Sequential(
            nn.Flatten(0,-1), #changed to start_dim=0 as this isn't running on batches !!! when running me on batches change this !!!
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size)
        )

    def forward(self, x):
        out = self.fully_connected(x)
        return out

Using cpu device


## Main setup

In [26]:
import subprocess
import tkinter

#get monitor resolution
root = tkinter.Tk()
width = root.winfo_screenwidth()
height = root.winfo_screenheight()
root.destroy() # Destroy the Tkinter window

#screen grab bounds
dims = np.array([155,86]) #game width, height
window_header = 11 #windows: remove title bar. Linux would need editing here.
origin = np.array([int(width/2 - dims[0]/2),int(height/2 - dims[1]/2)+window_header])

# #create frame grabber, dxcam. fastest but windows only.
region = (origin[0], origin[1], origin[0]+dims[0], origin[1]+dims[1])
frame_grabber = frame_dx(region)

#alternatively, mss frame grabber. slower but crossplatform.
# region = {"top":int(origin[1]), "left":int(origin[0]), "width":int(dims[0]), "height":int(dims[1])}    
# frame_grabber = frame_mss(region)

#create model
model = FullyConnected(dims).to(device)
print(model)

#start unity now!
unity_process = subprocess.Popen([r"C:\Users\BionicVisionVR\Documents\Mouse\starter_kits_robust_foraging\robust_foraging_windows_torch\Builds\RandomTrain\2D go to target v1.exe"])

#open socket
try:
    unity = tcpsocket(12345)
except socket.timeout:
    print('Connection to Unity timeout!')

You already created a DXCamera Instance for Device 0--Output 0!
Returning the existed instance...
To change capture parameters you can manually delete the old object using `del obj`.
FullyConnected(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fully_connected): Sequential(
    (0): Flatten(start_dim=0, end_dim=-1)
    (1): Linear(in_features=13330, out_features=512, bias=True)
    (2): ReLU()
    (3): Linear(in_features=512, out_features=512, bias=True)
    (4): ReLU()
    (5): Linear(in_features=512, out_features=3, bias=True)
  )
)


OSError: [WinError 10048] Only one usage of each socket address (protocol/network address/port) is normally permitted

In [ ]:
duration=1000 #make this as long as you want, or replace the iterator below with while true.

# Unity plots a few frames to get the screen updating, then waits for the blocking loop to start.
#
# 1. unity plots a frame, tells Python by sending the previous frame's reward, then waits for the next message
# 2. python reads that message, then grabs the frame and processes with the nn
# 3. python sends its next action to unity, then waits for the next message
# then the loop restarts. 
# 
# By the time this cell is started, step 1 is complete. Step 2 is the first action in the loop below.
# 
# The loop iterations are synced between python and unity. Both ends have blocking comms calls, so it stays synchronous.
# I've already validated that they stay synchronized in operation. 

now = time.perf_counter()

for i in range(duration):

    #wait for unity to flip a frame, grab the reward value from the last frame
    try:
        reward = unity.getresponse() #blocking
    except socket.timeout:
        print('Unity feedback timeout!')
        unity.close()
        break
    except Exception as e:
        print('Unexpected error:', e)
        break

    #grab a frame- this loops until windows flips to new frame
    frame = frame_grabber.grab() #this could also block for a very short period while waiting for windows to flip the screen.    
    # cv2.imshow('title',frame) #optionally plot the frame
    # cv2.waitKey(1)    
    
    #forward pass in model to get action
    input = torch.from_numpy(frame).float().to(device)
    with torch.no_grad():
        output = model(input)
    
    #send action to unity
    unity.sendvector(output.cpu().numpy())

    #for training: use "input", "action", and "reward" variables to construct (state,action,outcome)
    #note that you'll need input and action from the LAST iteration, plus reward from THIS iteration for an aligned sample

    #fps measurement
    if(i % 100 == 0 and i != 0):
        last = now
        now = time.perf_counter()
        print("fps:", 100 / (now-last)) #loops per second = process fps


# Clean up when finished.
print('Loop finished!')
frame_grabber.close()
unity.close()
del frame_grabber

unity_process.kill()

fps: 52.116669000624164
fps: 55.98584722570907
fps: 54.664514852703626
fps: 55.347268410284805
fps: 54.8182891573759
fps: 56.59282981296554
fps: 56.88087682554307
fps: 54.21630440416272
fps: 58.76747906438259
Loop finished!


In [ ]:
import subprocess
import tkinter

#get monitor resolution
root = tkinter.Tk()
width = root.winfo_screenwidth()
height = root.winfo_screenheight()
root.destroy() # Destroy the Tkinter window

#screen grab bounds
dims = np.array([155,86]) #game width, height
window_header = 11 #windows: remove title bar. Linux would need editing here.
origin = np.array([int(width/2 - dims[0]/2),int(height/2 - dims[1]/2)+window_header])

# #create frame grabber, dxcam. fastest but windows only.
region = (origin[0], origin[1], origin[0]+dims[0], origin[1]+dims[1])
frame_grabber = frame_dx(region)

#alternatively, mss frame grabber. slower but crossplatform.
# region = {"top":int(origin[1]), "left":int(origin[0]), "width":int(dims[0]), "height":int(dims[1])}    
# frame_grabber = frame_mss(region)

#create model
model = FullyConnected(dims).to(device)
print(model)

#start unity now!
unity_process = subprocess.Popen([r"C:\Users\BionicVisionVR\Documents\Mouse\starter_kits_robust_foraging\robust_foraging_windows_torch\Builds\RandomTrain\2D go to target v1.exe"])

#open socket
try:
    unity = tcpsocket(12345)
except socket.timeout:
    print('Connection to Unity timeout!')

FullyConnected(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fully_connected): Sequential(
    (0): Flatten(start_dim=0, end_dim=-1)
    (1): Linear(in_features=13330, out_features=512, bias=True)
    (2): ReLU()
    (3): Linear(in_features=512, out_features=512, bias=True)
    (4): ReLU()
    (5): Linear(in_features=512, out_features=3, bias=True)
  )
)
waiting for unity connection...
unity connected!


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

class CNNActorCritic(nn.Module):
    def __init__(self, in_channels=1, width=155, height=86, action_dim=3):
        super().__init__()
        # conv encoder
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),       nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),       nn.ReLU(),
        )
        # figure out conv output size
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, height, width)
            conv_out = self.conv(dummy).view(1, -1).size(1)
        # shared fully-connected feature
        self.fc = nn.Sequential(
            nn.Linear(conv_out, 512), nn.ReLU(),
        )
        # actor: mean and (learned) log-std
        self.mu_head = nn.Linear(512, action_dim)
        self.logstd = nn.Parameter(torch.zeros(action_dim))
        # critic
        self.value_head = nn.Linear(512, 1)

    def forward(self, x):
        # x: [B, H, W] or [B, 1, H, W]
        if x.dim() == 3:
            x = x.unsqueeze(1)
        features = self.conv(x)
        features = features.view(features.size(0), -1)
        h = self.fc(features)
        mu    = self.mu_head(h)
        std   = torch.exp(self.logstd)
        value = self.value_head(h).squeeze(-1)
        return mu, std, value

    def get_action_and_value(self, obs):
        mu, std, value = self.forward(obs)
        dist = torch.distributions.Normal(mu, std)
        action = dist.sample()
        logp = dist.log_prob(action).sum(-1)
        return action, logp, value

# --- minimal PPO loop skeleton ---
def ppo_update(model, optimizer, batch, clip_eps=0.2, vf_coef=0.5, ent_coef=0.01):
    obs, actions, logp_old, returns, advantages = batch
    mu, std, values = model(obs)
    dist = torch.distributions.Normal(mu, std)
    logp = dist.log_prob(actions).sum(-1)
    ratio = torch.exp(logp - logp_old)

    # policy loss
    pg_loss1 = ratio * advantages
    pg_loss2 = torch.clamp(ratio, 1-clip_eps, 1+clip_eps) * advantages
    policy_loss = -torch.min(pg_loss1, pg_loss2).mean()

    # value loss
    value_loss = F.mse_loss(values, returns)

    # entropy bonus
    entropy = dist.entropy().sum(-1).mean()

    loss = policy_loss + vf_coef * value_loss - ent_coef * entropy

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


In [ ]:
model = CNNActorCritic().to(device)
optimizer = optim.Adam(model.parameters(), lr=2.5e-4)

# placeholders for rollout storage
rollout = {"obs":[], "actions":[], "logp":[], "rewards":[], "values":[]}

batch_size=128        # mini-batch size for each gradient update  
T=2048        # number of steps to collect before each policy update  
lr=0.0003       # initial learning rate  
beta=0.005        # entropy regularization strength  
epsilon=0.2          # clipping parameter for PPO objective  
lamb=0.95         # GAE discount / advantage smoothing  
num_epoch=3            # how many passes over the buffer per update  
learning_rate_schedule='linear'  # anneal LR linearly to zero  
gamma=0.99         # reward discount factor  

num_steps= 10000

In [21]:
rollout = {"obs":[], "actions":[], "logp":[], "rewards":[], "values":[]}

for step in range(10000):
    # 1) receive reward from Unity
    try:
        reward = unity.getresponse()   # blocks up to 0.1 s
    except socket.timeout:
        reward = 0.0    
#    reward = unity.getresponse()  # shape [1]
    rollout["rewards"].append(torch.tensor(reward, device=device))

    # 2) grab frame, forward pass
    frame = frame_grabber.grab()
    obs = torch.from_numpy(frame).float().to(device) / 255.0
    obs = obs.unsqueeze(0).unsqueeze(0)
   
    action, logp, value = model.get_action_and_value(obs)
    rollout["obs"].append(obs.unsqueeze(0))
    rollout["actions"].append(action)
    rollout["logp"].append(logp)
    rollout["values"].append(value)

    # 3) send action back to Unity
    unity.sendvector(action.cpu().numpy())
    #fps measurement
    # if(step % 100 == 0 and step != 0):
    #     last = now
    #     now = time.perf_counter()
    #     print("fps:", 100 / (now-last)) #loops per second = process fps

    # every T steps, do a PPO update
    if (step + 1) % T == 0:
        # compute returns & advantages
        with torch.no_grad():
            frame = frame_grabber.grab()
            obs = torch.from_numpy(frame).float().to(device) / 255.0
            obs = obs.unsqueeze(0).unsqueeze(0)        
            _, _, next_value = model.get_action_and_value(obs)
        returns = []
        gae = 0
        for i in reversed(range(T)):
            delta = rollout["rewards"][i] + gamma * (next_value if i == T-1 else rollout["values"][i+1]) - rollout["values"][i]
            gae = delta + gamma * lam * gae
            returns.insert(0, gae + rollout["values"][i])
        obs_batch      = torch.cat(rollout["obs"])
        actions_batch  = torch.stack(rollout["actions"])
        logp_batch     = torch.stack(rollout["logp"])
        returns_batch  = torch.stack(returns).detach()
        advantages_batch = returns_batch - torch.stack(rollout["values"])
        # normalize advantages
        advantages_batch = (advantages_batch - advantages_batch.mean()) / (advantages_batch.std() + 1e-8)
        batch = (obs_batch, actions_batch, logp_batch, returns_batch, advantages_batch)
        ppo_update(model, optimizer, batch)

        print('Step:', step,'reward:',reward)
        # clear rollout
        for k in rollout: rollout[k].clear()

# clean up as before...


ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host

In [ ]:
# duration=1000 #make this as long as you want, or replace the iterator below with while true.

# # Unity plots a few frames to get the screen updating, then waits for the blocking loop to start.
# #
# # 1. unity plots a frame, tells Python by sending the previous frame's reward, then waits for the next message
# # 2. python reads that message, then grabs the frame and processes with the nn
# # 3. python sends its next action to unity, then waits for the next message
# # then the loop restarts. 
# # 
# # By the time this cell is started, step 1 is complete. Step 2 is the first action in the loop below.
# # 
# # The loop iterations are synced between python and unity. Both ends have blocking comms calls, so it stays synchronous.
# # I've already validated that they stay synchronized in operation. 

# now = time.perf_counter()

# for i in range(duration):

#     #wait for unity to flip a frame, grab the reward value from the last frame
#     try:
#         reward = unity.getresponse() #blocking
#     except socket.timeout:
#         print('Unity feedback timeout!')
#         unity.close()
#         break
#     except Exception as e:
#         print('Unexpected error:', e)
#         break

#     #grab a frame- this loops until windows flips to new frame
#     frame = frame_grabber.grab() #this could also block for a very short period while waiting for windows to flip the screen.    
#     # cv2.imshow('title',frame) #optionally plot the frame
#     # cv2.waitKey(1)    
    
#     #forward pass in model to get action
#     input = torch.from_numpy(frame).float().to(device)
#     with torch.no_grad():
#         output = model(input)
    
#     #send action to unity
#     unity.sendvector(output.cpu().numpy())

#     #for training: use "input", "action", and "reward" variables to construct (state,action,outcome)
#     #note that you'll need input and action from the LAST iteration, plus reward from THIS iteration for an aligned sample

#     #fps measurement
#     if(i % 100 == 0 and i != 0):
#         last = now
#         now = time.perf_counter()
#         print("fps:", 100 / (now-last)) #loops per second = process fps


# # Clean up when finished.
# print('Loop finished!')
# frame_grabber.close()
# unity.close()
# del frame_grabber

# unity_process.kill()

In [20]:
model = CNNActorCritic().to(device)
optimizer = optim.Adam(model.parameters(), lr=2.5e-4)

# placeholders for rollout storage
rollout = {"obs":[], "actions":[], "logp":[], "rewards":[], "values":[]}

batch_size=128        # mini-batch size for each gradient update  
T=2048        # number of steps to collect before each policy update  
lr=0.0003       # initial learning rate  
beta=0.005        # entropy regularization strength  
epsilon=0.2          # clipping parameter for PPO objective  
lamb=0.95         # GAE discount / advantage smoothing  
num_epoch=3            # how many passes over the buffer per update  
learning_rate_schedule='linear'  # anneal LR linearly to zero  
gamma=0.99         # reward discount factor  

num_steps= 10000

In [ ]:
import socket
import json
import time

HOST, PORT = "127.0.0.1", 5005

# 1) connect to Unity’s pythonController.cs TCP listener
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
sock.connect((HOST, PORT))
sock.settimeout(1.0)

# 2) read Unity’s “ready” banner (if any)
try:
    banner = sock.recv(1024).decode("utf-8")
    print("Unity says:", banner)
except socket.timeout:
    pass

# 3) send a reset message (if your C# side supports it)
#    -- if not, just skip to step 4
reset_msg = json.dumps({"cmd": "reset"}) + "\n"
sock.sendall(reset_msg.encode("utf-8"))
time.sleep(0.1)

# 4) now loop 10 times, blasting Unity with a nonzero action
for step in range(10):
    # arbitrary action vector: here two floats in [–1, +1]
    action = [0.8, -0.3]
    msg = {"cmd":"act", "action": action}
    sock.sendall((json.dumps(msg) + "\n").encode("utf-8"))

    # flush & wait a tiny bit
    time.sleep(0.05)

    # try to read back whatever Unity sends (obs, reward, done…)
    try:
        data = sock.recv(4096).decode("utf-8")
        print(f"step {step:2d} ←", data.strip())
    except socket.timeout:
        print(f"step {step:2d} → no response (timeout)")


ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

In [ ]:
rollout = {"obs":[], "actions":[], "logp":[], "rewards":[], "values":[]}

for step in range(10000):
    # 1) receive reward from Unity
    try:
        reward = unity.getresponse()   # blocks up to 0.1 s
    except socket.timeout:
        reward = 0.0    
#    reward = unity.getresponse()  # shape [1]
    rollout["rewards"].append(torch.tensor(reward, device=device))

    # 2) grab frame, forward pass
    frame = frame_grabber.grab()
    obs = torch.from_numpy(frame).float().to(device) / 255.0
    obs = obs.unsqueeze(0).unsqueeze(0)
   
    action, logp, value = model.get_action_and_value(obs)
    rollout["obs"].append(obs.unsqueeze(0))
    rollout["actions"].append(action)
    rollout["logp"].append(logp)
    rollout["values"].append(value)
    action[:] =10
    #action
    # 3) send action back to Unity
    unity.sendvector(action.cpu().numpy())

In [ ]:
try:
    reward = unity.getresponse()   # blocks up to 0.1 s
except socket.timeout:
    reward = 0.0

In [ ]:
# 2) grab frame, forward pass
frame = frame_grabber.grab()
obs = torch.from_numpy(frame).float().to(device) / 255.0
obs = obs.unsqueeze(0).unsqueeze(0)

action, logp, value = model.get_action_and_value(obs)

rollout["obs"].append(obs.unsqueeze(0))
rollout["actions"].append(action)
rollout["logp"].append(logp)
rollout["values"].append(value)

In [ ]:
unity.sendvector(action.cpu().numpy())

In [ ]:
action

tensor([[-0.4003,  0.4869,  0.7359]])

In [ ]:
with torch.no_grad():
    frame = frame_grabber.grab()
    obs = torch.from_numpy(frame).float().to(device) / 255.0
    obs = obs.unsqueeze(0).unsqueeze(0)        
    _, _, next_value = model.get_action_and_value(obs)

In [ ]:
returns = []
gae = 0
for i in reversed(range(T)):
    delta = rollout["rewards"][i] + gamma * (next_value if i == T-1 else rollout["values"][i+1]) - rollout["values"][i]
    gae = delta + gamma * lamb * gae
    returns.insert(0, gae + rollout["values"][i])

IndexError: list index out of range

In [ ]:
rollout = {"obs":[], "actions":[], "logp":[], "rewards":[], "values":[]}

for step in range(10000):
    # 1) receive reward from Unity
    try:
        reward = unity.getresponse()   # blocks up to 0.1 s
    except socket.timeout:
        reward = 0.0    
#    reward = unity.getresponse()  # shape [1]
    rollout["rewards"].append(torch.tensor(reward, device=device))

    # 2) grab frame, forward pass
    frame = frame_grabber.grab()
    obs = torch.from_numpy(frame).float().to(device) / 255.0
    obs = obs.unsqueeze(0).unsqueeze(0)
   
    action, logp, value = model.get_action_and_value(obs)
    rollout["obs"].append(obs.unsqueeze(0))
    rollout["actions"].append(action)
    rollout["logp"].append(logp)
    rollout["values"].append(value)

    # 3) send action back to Unity
    unity.sendvector(action.cpu().numpy())
    #fps measurement
    # if(step % 100 == 0 and step != 0):
    #     last = now
    #     now = time.perf_counter()
    #     print("fps:", 100 / (now-last)) #loops per second = process fps

    # every T steps, do a PPO update
    if (step + 1) % T == 0:
        # compute returns & advantages
        with torch.no_grad():
            frame = frame_grabber.grab()
            obs = torch.from_numpy(frame).float().to(device) / 255.0
            obs = obs.unsqueeze(0).unsqueeze(0)        
            _, _, next_value = model.get_action_and_value(obs)
        returns = []
        gae = 0
        for i in reversed(range(T)):
            delta = rollout["rewards"][i] + gamma * (next_value if i == T-1 else rollout["values"][i+1]) - rollout["values"][i]
            gae = delta + gamma * lam * gae
            returns.insert(0, gae + rollout["values"][i])
        obs_batch      = torch.cat(rollout["obs"])
        actions_batch  = torch.stack(rollout["actions"])
        logp_batch     = torch.stack(rollout["logp"])
        returns_batch  = torch.stack(returns).detach()
        advantages_batch = returns_batch - torch.stack(rollout["values"])
        # normalize advantages
        advantages_batch = (advantages_batch - advantages_batch.mean()) / (advantages_batch.std() + 1e-8)
        batch = (obs_batch, actions_batch, logp_batch, returns_batch, advantages_batch)
        ppo_update(model, optimizer, batch)

        print('Step:', step,'reward:',reward)
        # clear rollout
        for k in rollout: rollout[k].clear()

# clean up as before...


NameError: name 'lam' is not defined

In [ ]:
#grab a frame- this loops until windows flips to new frame
frame = frame_grabber.grab() #this could also block for a very short period while waiting for windows to flip the screen.    
# cv2.imshow('title',frame) #optionally plot the frame
# cv2.waitKey(1)    

#forward pass in model to get action
input = torch.from_numpy(frame).float().to(device)
input = input.unsqueeze(0).unsqueeze(0)

with torch.no_grad():
    output = model(input)

#send action to unity
unity.sendvector(output.cpu().numpy())

AttributeError: 'tuple' object has no attribute 'cpu'

In [ ]:
reward = unity.getresponse()  # shape [1]
rollout["rewards"].append(torch.tensor(reward, device=device))

# 2) grab frame, forward pass
frame = frame_grabber.grab()
obs = torch.from_numpy(frame).float().to(device) / 255.0
obs = obs.unsqueeze(0).unsqueeze(0)

action, logp, value = model.get_action_and_value(obs)
rollout["obs"].append(obs.unsqueeze(0))
rollout["actions"].append(action)
rollout["logp"].append(logp)
rollout["values"].append(value)

# 3) send action back to Unity
unity.sendvector(action.cpu().numpy())

In [ ]:
rollout = {"obs":[], "actions":[], "logp":[], "rewards":[], "values":[]}

for step in range(10000):
    # 1) receive reward from Unity
    reward = unity.getresponse()  # shape [1]
    rollout["rewards"].append(torch.tensor(reward, device=device))

    # 2) grab frame, forward pass
    frame = frame_grabber.grab()
    obs = torch.from_numpy(frame).float().to(device) / 255.0
    action, logp, value = model.get_action_and_value(obs)
    rollout["obs"].append(obs.unsqueeze(0))
    rollout["actions"].append(action)
    rollout["logp"].append(logp)
    rollout["values"].append(value)

    # 3) send action back to Unity
    unity.sendvector(action.cpu().numpy())

    # every T steps, do a PPO update
    if (step + 1) % T == 0:
        # compute returns & advantages
        with torch.no_grad():
            _, _, next_value = model.get_action_and_value(torch.from_numpy(frame_grabber.grab()).float().to(device)/255.0)
        returns = []
        gae = 0
        for i in reversed(range(T)):
            delta = rollout["rewards"][i] + gamma * (next_value if i == T-1 else rollout["values"][i+1]) - rollout["values"][i]
            gae = delta + gamma * lam * gae
            returns.insert(0, gae + rollout["values"][i])
        obs_batch      = torch.cat(rollout["obs"])
        actions_batch  = torch.stack(rollout["actions"])
        logp_batch     = torch.stack(rollout["logp"])
        returns_batch  = torch.stack(returns).detach()
        advantages_batch = returns_batch - torch.stack(rollout["values"])
        # normalize advantages
        advantages_batch = (advantages_batch - advantages_batch.mean()) / (advantages_batch.std() + 1e-8)
        batch = (obs_batch, actions_batch, logp_batch, returns_batch, advantages_batch)
        ppo_update(model, optimizer, batch)

        # clear rollout
        for k in rollout: rollout[k].clear()

# clean up as before...


RuntimeError: Expected 3D (unbatched) or 4D (batched) input to conv2d, but got input of size: [86, 155]

In [ ]:
duration=1000 #make this as long as you want, or replace the iterator below with while true.

# Unity plots a few frames to get the screen updating, then waits for the blocking loop to start.
#
# 1. unity plots a frame, tells Python by sending the previous frame's reward, then waits for the next message
# 2. python reads that message, then grabs the frame and processes with the nn
# 3. python sends its next action to unity, then waits for the next message
# then the loop restarts. 
# 
# By the time this cell is started, step 1 is complete. Step 2 is the first action in the loop below.
# 
# The loop iterations are synced between python and unity. Both ends have blocking comms calls, so it stays synchronous.
# I've already validated that they stay synchronized in operation. 

now = time.perf_counter()

for i in range(duration):

    #wait for unity to flip a frame, grab the reward value from the last frame
    try:
        reward = unity.getresponse() #blocking
    except socket.timeout:
        print('Unity feedback timeout!')
        unity.close()
        break
    except Exception as e:
        print('Unexpected error:', e)
        break

    #grab a frame- this loops until windows flips to new frame
    frame = frame_grabber.grab() #this could also block for a very short period while waiting for windows to flip the screen.    
    # cv2.imshow('title',frame) #optionally plot the frame
    # cv2.waitKey(1)    
    
    #forward pass in model to get action
    input = torch.from_numpy(frame).float().to(device)
    with torch.no_grad():
        output = model(input)
    
    #send action to unity
    unity.sendvector(output.cpu().numpy())

    #for training: use "input", "action", and "reward" variables to construct (state,action,outcome)
    #note that you'll need input and action from the LAST iteration, plus reward from THIS iteration for an aligned sample

    #fps measurement
    if(i % 100 == 0 and i != 0):
        last = now
        now = time.perf_counter()
        print("fps:", 100 / (now-last)) #loops per second = process fps


# Clean up when finished.
print('Loop finished!')
frame_grabber.close()
unity.close()
del frame_grabber

unity_process.kill()

In [ ]:
input

In [ ]:

# --- putting it all together in your VR_RL.py loop ---
if __name__ == "__main__":
    model = CNNActorCritic().to(device)
    optimizer = optim.Adam(model.parameters(), lr=2.5e-4)

    # placeholders for rollout storage
    rollout = {"obs":[], "actions":[], "logp":[], "rewards":[], "values":[]}

    for step in range(num_steps):
        # 1) receive reward from Unity
        reward = unity.getresponse()  # shape [1]
        rollout["rewards"].append(torch.tensor(reward, device=device))

        # 2) grab frame, forward pass
        frame = frame_grabber.grab()
        obs = torch.from_numpy(frame).float().to(device) / 255.0
        action, logp, value = model.get_action_and_value(obs)
        rollout["obs"].append(obs.unsqueeze(0))
        rollout["actions"].append(action)
        rollout["logp"].append(logp)
        rollout["values"].append(value)

        # 3) send action back to Unity
        unity.sendvector(action.cpu().numpy())

        # every T steps, do a PPO update
        if (step + 1) % T == 0:
            # compute returns & advantages
            with torch.no_grad():
                _, _, next_value = model.get_action_and_value(torch.from_numpy(frame_grabber.grab()).float().to(device)/255.0)
            returns = []
            gae = 0
            for i in reversed(range(T)):
                delta = rollout["rewards"][i] + gamma * (next_value if i == T-1 else rollout["values"][i+1]) - rollout["values"][i]
                gae = delta + gamma * lam * gae
                returns.insert(0, gae + rollout["values"][i])
            obs_batch      = torch.cat(rollout["obs"])
            actions_batch  = torch.stack(rollout["actions"])
            logp_batch     = torch.stack(rollout["logp"])
            returns_batch  = torch.stack(returns).detach()
            advantages_batch = returns_batch - torch.stack(rollout["values"])
            # normalize advantages
            advantages_batch = (advantages_batch - advantages_batch.mean()) / (advantages_batch.std() + 1e-8)
            batch = (obs_batch, actions_batch, logp_batch, returns_batch, advantages_batch)
            ppo_update(model, optimizer, batch)

            # clear rollout
            for k in rollout: rollout[k].clear()

    # clean up as before...


## Main loop

In [ ]:
duration=1000 #make this as long as you want, or replace the iterator below with while true.

# Unity plots a few frames to get the screen updating, then waits for the blocking loop to start.
#
# 1. unity plots a frame, tells Python by sending the previous frame's reward, then waits for the next message
# 2. python reads that message, then grabs the frame and processes with the nn
# 3. python sends its next action to unity, then waits for the next message
# then the loop restarts. 
# 
# By the time this cell is started, step 1 is complete. Step 2 is the first action in the loop below.
# 
# The loop iterations are synced between python and unity. Both ends have blocking comms calls, so it stays synchronous.
# I've already validated that they stay synchronized in operation. 

now = time.perf_counter()

for i in range(duration):

    #wait for unity to flip a frame, grab the reward value from the last frame
    try:
        reward = unity.getresponse() #blocking
    except socket.timeout:
        print('Unity feedback timeout!')
        unity.close()
        break
    except Exception as e:
        print('Unexpected error:', e)
        break

    #grab a frame- this loops until windows flips to new frame
    frame = frame_grabber.grab() #this could also block for a very short period while waiting for windows to flip the screen.    
    # cv2.imshow('title',frame) #optionally plot the frame
    # cv2.waitKey(1)    
    
    #forward pass in model to get action
    input = torch.from_numpy(frame).float().to(device)
    with torch.no_grad():
        output = model(input)
    
    #send action to unity
    unity.sendvector(output.cpu().numpy())

    #for training: use "input", "action", and "reward" variables to construct (state,action,outcome)
    #note that you'll need input and action from the LAST iteration, plus reward from THIS iteration for an aligned sample

    #fps measurement
    if(i % 100 == 0 and i != 0):
        last = now
        now = time.perf_counter()
        print("fps:", 100 / (now-last)) #loops per second = process fps


# Clean up when finished.
print('Loop finished!')
frame_grabber.close()
unity.close()
del frame_grabber

unity_process.kill()

fps: 55.63065694242782
fps: 59.221536456570604
fps: 83.34207730627746
fps: 62.99616964389708
fps: 61.42402763000178
fps: 50.67790053823476
fps: 67.1868812508155
Unexpected error: Empty message from Unity, socket disconnected
Loop finished!


Now unity can be closed.

## Notes and examples

### Benchmarks for mss and dxcam

In [ ]:
def screen_record_mss() -> int:
    mon = {"top": 0, "left": 40, "width": 155, "height": 80}

    title = "[MSS] FPS benchmark"
    start_time, fps = time.perf_counter(), 0
    sct = mss.mss()
    start = time.perf_counter()

    while fps < 1000:
        img = np.asarray(sct.grab(mon))
        fps += 1

        cv2.imshow(title, img)
        cv2.waitKey(1)
    
    end_time = time.perf_counter() - start_time
    print(f"{fps/end_time}")
    cv2.destroyAllWindows()

    return fps


def screen_record_dxcam() -> int:
    start_time, fps = time.perf_counter(), 0
    region = (0, 40, 155, 126)
    cam = dxcam.create(region=region)
    start = time.perf_counter()
    title = "dxcam benchmark"
    while fps < 1000:
        frame = cam.grab()
        if frame is not None:  # only count new frames
            fps += 1
            cv2.imshow(title,frame)
            cv2.waitKey(1)

    end_time = time.perf_counter() - start_time
    print(f"{fps/end_time}")
    cv2.destroyAllWindows()
    cam.release()
    print(type(frame))


In [ ]:
screen_record_mss()

In [ ]:
screen_record_dxcam()

#### Use cases for grabbers

dxcam

In [ ]:
region = (0, 40, 155, 126)
frame_grabber = frame_dx(region)
frame_grabber.grab()
frame_grabber.close()

mss

In [ ]:
region = {"top": 0, "left": 40, "width": 155, "height": 80}
frame_grabber = frame_mss(region)
frame_grabber.grab()
frame_grabber.close()

### Use cases for pytorch model

Create model

In [ ]:
model = FullyConnected(10).to(device)
print(model)

Run a sample through it

In [ ]:
frame = np.random.rand(28,28)

with torch.no_grad():

    input = torch.from_numpy(frame).float().to('cuda:0')

    #forward pass
    output = model(input)

print(output)